# Lekcija 33: Optimizacija

Lekcije 31–32 koristile su obični gradijentni spust: uzeti gradijent i napraviti korak fiksne veličine u suprotnom smjeru:

$$p \leftarrow p - \eta\,\nabla f(p)$$

gdje je $p$ ono što učimo (težina, pomak ili cijeli vektor parametara odjednom), $\eta$ je **stopa učenja**, a $\nabla f(p)$ gradijent gubitka pri trenutačnim parametrima. To je dovoljno za dokaz da *ideja* radi, ali stvarne mreže rijetko se tako treniraju. Ova lekcija obrađuje četiri praktična poboljšanja &mdash; **momentum**, **prilagodljive veličine koraka (Adam)**, **postupke inicijalizacije težina** i **rasporede stope učenja** &mdash; od kojih svako ispravlja određen, konkretan način neuspjeha običnog gradijentnog spusta.

Primijetite da se pri stvarnom treniranju gradijent gotovo nikada ne računa preko *cijelog* skupa za treniranje odjednom — ne bi stao u memoriju. Umjesto toga pri svakom se koraku procjenjuje iz male, nasumično uzorkovane **mini-serije** (primjerice 32 ili 256 primjera). Ta procjena sa šumom, koja varira od uzorka do uzorka, daje naziv **stohastičkom gradijentnom spustu (SGD)**. Ova lekcija uglavnom koristi točan gradijent fiksne funkcije bez šuma (radi jednostavnosti naziva ga „SGD”); posljednji odjeljak izravno simulira šum mini-serija kako bi pokazao zašto je važan.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

## Krajolik gubitka koji otežava obični gradijentni spust

$f(x,y) = 0.05x^2 + 5y^2$ je zdjela strma u smjeru $y$, a plitka u smjeru $x$. Veličina koraka dovoljna za napredak duž $x$ previše preskače duž $y$, stvarajući klasično *cik-cak* kretanje.

In [ ]:
def f(p):
    return 0.05 * p[0]**2 + 5 * p[1]**2

def grad_f(p):
    return np.array([0.1 * p[0], 10 * p[1]])

start = np.array([-4.0, 1.0])
lr = 0.15

p = start.copy()
sgd_path = [p.copy()]
for _ in range(30):
    p = p - lr * grad_f(p)
    sgd_path.append(p.copy())
sgd_path = np.array(sgd_path)

print(f'plain SGD, 30 steps: final loss = {f(sgd_path[-1]):.4f}')

In [ ]:
xs = np.linspace(-4.5, 1, 200)
ys = np.linspace(-1.5, 1.5, 200)
XX, YY = np.meshgrid(xs, ys)
ZZ = 0.05 * XX**2 + 5 * YY**2

plt.contour(XX, YY, ZZ, levels=20, colors='lightgray', linewidths=0.7)
plt.plot(*sgd_path.T, '-o', markersize=3, color='tab:blue')
plt.scatter([0], [0], marker='*', s=150, color='black', zorder=5, label='minimum')
plt.legend(fontsize=8)
plt.title('Plain gradient descent: the classic zigzag')
plt.show()

## Momentum: zapamtite kamo ste se kretali

Umjesto koraka isključivo duž trenutačnog gradijenta, akumulirajte tekuću **brzinu** &mdash; ponderirani prosjek prošlih gradijenata &mdash; i napravite korak duž nje:

$$v \leftarrow \beta v - \eta\,\nabla f(p), \qquad p \leftarrow p + v$$

Stvarna korist momentuma jest ubrzanje, a ne prigušenje. Duž dosljednog smjera (poput ovdje plitke osi $x$) svi gradijenti pokazuju jednako, pa tekuća brzina raste iz koraka u korak &mdash; efektivna veličina koraka duž te osi postaje mnogo veća nego što bi je dao sam $\eta$, upravo kao lopta koja ubrzava kotrljajući se nizbrdo. Duž oscilirajućeg smjera (strme osi $y$, koja mijenja predznak u svakom koraku) uzastopni se gradijenti djelomično poništavaju u istom tekućem prosjeku &mdash; ali samo *djelomično*: jedan koeficijent momentuma $\beta$ dijeli se među svim smjerovima, pa poništavanje nije prilagođeno zakrivljenosti pojedinog smjera. Zato strma os i dalje može znatno oscilirati &mdash; ponekad i više od samog običnog gradijentnog spusta &mdash; prije nego što se oscilacije smanje. Ukupni učinak tijekom cijelog postupka obično je ipak povoljan: brži napredak duž dobro ponašane osi nadmašuje nedovoljno prigušenje druge. No to je dobitak od ubrzanja uz grubu, neprilagođenu kočnicu, a ne potpuno rješenje osciliranja &mdash; za to je zapravo zamišljen Adam, koji slijedi.

(Momentum ima i drugu korist, još važniju pri stvarnom treniranju: uz bučne gradijente mini-serija umjesto ovdje korištenog točnog gradijenta, tekući prosjek djeluje kao niskopropusni filtar, zaglađujući šum između uzoraka u pouzdaniji smjer. Slučaj sa šumom izravno ćemo vidjeti pri kraju lekcije.)

In [ ]:
momentum = 0.9
p = start.copy()
v = np.zeros(2)
mom_path = [p.copy()]
for _ in range(30):
    g = grad_f(p)
    v = momentum * v - lr * g
    p = p + v
    mom_path.append(p.copy())
mom_path = np.array(mom_path)

print(f'SGD + momentum, 30 steps: final loss = {f(mom_path[-1]):.4f}')

### Provjera ispravnosti pomoću PyTorcha

In [ ]:
p_t = torch.tensor(start.copy(), requires_grad=True)
opt = torch.optim.SGD([p_t], lr=lr, momentum=momentum)
for _ in range(30):
    opt.zero_grad()
    loss = 0.05 * p_t[0]**2 + 5 * p_t[1]**2
    loss.backward()
    opt.step()

print(f'our result:   {mom_path[-1]}')
print(f'torch result: {p_t.detach().numpy()}')
print(f'max diff: {np.abs(p_t.detach().numpy() - mom_path[-1]).max():.2e}')

## Adam: prilagodljiva veličina koraka za svaki parametar

**Adam** (<a href="../references.html#kingma-ba-2014">Kingma & Ba, 2014.</a><span class="landmark-paper">&#9733;</span>) prati i tekući prosjek gradijenta nalik momentumu ($m$) *i* tekući prosjek kvadrata gradijenta ($v$), a zatim dijeli korak s $\sqrt{v}$ &mdash; automatski smanjujući veličinu koraka za parametre s dosljedno velikim gradijentima (poput ovdje strmog smjera $y$) i povećavajući je za parametre s malim gradijentima:

$$m \leftarrow \beta_1 m + (1-\beta_1)g, \qquad v \leftarrow \beta_2 v + (1-\beta_2)g^2, \qquad p \leftarrow p - \eta\frac{\hat{m}}{\sqrt{\hat{v}}+\epsilon}$$

($\hat{m}, \hat{v}$ su inačice $m, v$ s ispravljenom pristranošću, važne uglavnom u prvih nekoliko koraka.)

In [ ]:
beta1, beta2, adam_eps = 0.9, 0.999, 1e-8
adam_lr = 0.3

p = start.copy()
m, v_sq = np.zeros(2), np.zeros(2)
adam_path = [p.copy()]
for t in range(1, 31):
    g = grad_f(p)
    m = beta1 * m + (1 - beta1) * g
    v_sq = beta2 * v_sq + (1 - beta2) * g**2
    m_hat = m / (1 - beta1**t)
    v_hat = v_sq / (1 - beta2**t)
    p = p - adam_lr * m_hat / (np.sqrt(v_hat) + adam_eps)
    adam_path.append(p.copy())
adam_path = np.array(adam_path)

p_t2 = torch.tensor(start.copy(), requires_grad=True)
opt2 = torch.optim.Adam([p_t2], lr=adam_lr, betas=(beta1, beta2), eps=adam_eps)
for _ in range(30):
    opt2.zero_grad()
    loss = 0.05 * p_t2[0]**2 + 5 * p_t2[1]**2
    loss.backward()
    opt2.step()

print(f'Adam, 30 steps: final loss = {f(adam_path[-1]):.4f}')
print(f'max diff vs torch.optim.Adam: {np.abs(p_t2.detach().numpy() - adam_path[-1]).max():.2e}')

### Prikaz svih triju putanja

In [ ]:
plt.contour(XX, YY, ZZ, levels=20, colors='lightgray', linewidths=0.7)
plt.plot(*sgd_path.T, '-o', markersize=3, label=f'plain SGD (loss={f(sgd_path[-1]):.3f})')
plt.plot(*mom_path.T, '-o', markersize=3, label=f'momentum (loss={f(mom_path[-1]):.3f})')
plt.plot(*adam_path.T, '-o', markersize=3, label=f'Adam (loss={f(adam_path[-1]):.3f})')
plt.scatter([0], [0], marker='*', s=150, color='black', zorder=5, label='minimum')
plt.legend(fontsize=8)
plt.title('30 steps of each optimizer on the same narrow bowl')
plt.show()

Obični SGD ostaje zarobljen u osciliranju preko uske doline, jedva napredujući duž plitkog smjera. Akumulirana brzina momentuma jasno ubrzava napredak duž tog smjera i završava manjim gubitkom od običnog SGD-a &mdash; ali primijetite da njegova putanja zapravo oscilira *šire* u $y$ nego putanja običnog SGD-a, a ne uže, jer jedan zajednički koeficijent momentuma ne može svaku os prigušiti baš koliko je njoj potrebno. Adam, koji svaki smjer neovisno skalira, doista izravno obuzdava strmu os, a istodobno brzo napreduje duž plitke &mdash; postižući najmanji gubitak od triju metoda u istih 30 koraka.

## Inicijalizacija težina: zašto nula ne radi

Primamljivo je sve težine inicijalizirati na nulu. Iako se to čini „najneutralnijim” polazištem, zapravo je pogubno za svaki sloj s više od jedne jedinice: svaka skrivena jedinica računa potpuno istu funkciju ulaza, dobiva potpuno isti gradijent i ažurira se za potpuno isti iznos, zauvijek. Ta se **simetrija** nikada ne narušava sama od sebe.

In [ ]:
rng = np.random.default_rng(0)
theta_in = rng.uniform(0, 2 * np.pi, 60)
inner = np.stack([0.5 * np.cos(theta_in), 0.5 * np.sin(theta_in)], axis=1) + rng.normal(0, 0.1, (60, 2))
theta_out = rng.uniform(0, 2 * np.pi, 60)
outer = np.stack([2.0 * np.cos(theta_out), 2.0 * np.sin(theta_out)], axis=1) + rng.normal(0, 0.15, (60, 2))
X = np.vstack([inner, outer])
y = np.concatenate([np.zeros(60), np.ones(60)])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(np.float64)

H = 4

def train_mlp(W1, b1, W2, b2, n_epochs=3000, lr=0.1):
    for _ in range(n_epochs):
        z1 = X @ W1 + b1
        a1 = relu(z1)
        z2 = (a1 @ W2 + b2).ravel()
        p = sigmoid(z2)
        n = len(y)
        grad_z2 = ((p - y) / n).reshape(-1, 1)
        grad_W2 = a1.T @ grad_z2
        grad_b2 = grad_z2.sum(axis=0)
        grad_a1 = grad_z2 @ W2.T
        grad_z1 = grad_a1 * relu_deriv(z1)
        grad_W1 = X.T @ grad_z1
        grad_b1 = grad_z1.sum(axis=0)
        W2 -= lr * grad_W2; b2 -= lr * grad_b2
        W1 -= lr * grad_W1; b1 -= lr * grad_b1
    return W1, b1, W2, b2, p

W1_zero, b1_zero = np.zeros((2, H)), np.zeros(H)
W2_zero, b2_zero = np.zeros((H, 1)), np.zeros(1)
W1_zero, b1_zero, W2_zero, b2_zero, p_zero = train_mlp(W1_zero, b1_zero, W2_zero, b2_zero)
acc_zero = ((p_zero > 0.5).astype(float) == y).mean()

rng2 = np.random.default_rng(8)
W1_rand = rng2.normal(size=(2, H)) * 0.7
W2_rand = rng2.normal(size=(H, 1)) * 0.7
b1_rand, b2_rand = np.zeros(H), np.zeros(1)
_, _, _, _, p_rand = train_mlp(W1_rand, b1_rand, W2_rand, b2_rand)
acc_rand = ((p_rand > 0.5).astype(float) == y).mean()

print(f'zero-initialized:   accuracy = {acc_zero:.1%}')
print(f'randomly initialized: accuracy = {acc_rand:.1%}')
print()
print('all 4 hidden units still have identical weight vectors after 3000 steps of zero-init training:')
print(np.round(W1_zero, 4))

Uz inicijalizaciju nulama sve četiri skrivene jedinice zauvijek ostaju na točno nuli &mdash; model nikada ne izlazi iz trivijalnog rješenja (na razini slučajnog pogađanja), bez obzira na trajanje treniranja. Mala *nasumična* inicijalizacija narušava simetriju: svaka jedinica u početku računa nešto malo drukčije, pa ih gradijentni spust može razdvojiti i omogućiti im specijalizaciju. Zato zadana inicijalizacija slojeva u svakom razvojnom okviru koristi male nasumične vrijednosti, nikada nule.

## Postupci inicijalizacije težina: koliko trebaju biti „male nasumične” vrijednosti?

Inicijalizacija nulama potpuno ne uspijeva; „male nasumične” vrijednosti to ispravljaju. Ali *koliko* male? Odaberite fiksnu standardnu devijaciju i radit će za jednu širinu sloja, a zatim tiho zakazati za drugu. Pratimo varijancu aktivacija nakon prolaska nasumičnih podataka kroz dubok niz linearnih slojeva s aktivacijom `tanh`, koristeći istu fiksnu standardnu devijaciju težina pri svakoj širini:

In [ ]:
def forward_variance(width, std, depth=20):
    x = torch.randn(100, width)
    for _ in range(depth):
        W = torch.randn(width, width) * std
        x = torch.tanh(x @ W)
    return x.var().item()

print('fixed weight std = 0.05, activation variance after 20 tanh layers:')
for width in [16, 64, 256]:
    print(f'  width={width:4d}: variance = {forward_variance(width, std=0.05):.2e}')

Uz fiksnu standardnu devijaciju 0.05 signal nakon 20 slojeva pada praktički na nulu — a jačina tog pada snažno ovisi o širini sloja. Varijanca prije aktivacije svakog sloja skalira se kao `fan_in * std^2`, gdje je **fan-in** broj ulaza sloja: pri širini 16 to je `16 * 0.05^2 ≈ 0.04`, pa svaki sloj zadržava samo oko 4 % varijance prethodnog, a 20 uzastopnih slojeva gotovo potpuno briše signal. Pri širini 256 *ista* fiksna standardna devijacija zadržava `256 * 0.05^2 ≈ 0.64` varijance po sloju — i dalje se smanjuje, ali mnogo sporije, pa zato gornji stupac širine 256 pokazuje znatno manje nestajanje nego onaj širine 16. Učinak fiksne standardne devijacije potpuno ovisi o broju ulaza, pa bi ista vrijednost koja snažno smanjuje signal u uskom sloju, pri dovoljno širokom sloju, povisila `fan_in * std^2` iznad 1 i umjesto toga pokrenula *eksplozivan rast*.

**Xavierova inicijalizacija** (<a href="../references.html#glorot-bengio-2010">Glorot & Bengio, 2010.</a>) &mdash; poznata i kao *Glorotova inicijalizacija* &mdash; to ispravlja skaliranjem standardne devijacije prema broju ulaza sloja: `std = sqrt(1 / fan_in)`, čime `fan_in * std^2` ostaje točno 1 bez obzira na širinu. Ponovimo isti eksperiment s težinama skaliranima prema Xavieru:

In [ ]:
def forward_variance_xavier(width, depth=20):
    x = torch.randn(100, width)
    for _ in range(depth):
        std = (1.0 / width) ** 0.5
        W = torch.randn(width, width) * std
        x = torch.tanh(x @ W)
    return x.var().item()

print('Xavier std = sqrt(1/fan_in), activation variance after 20 tanh layers:')
for width in [16, 64, 256]:
    print(f'  width={width:4d}: variance = {forward_variance_xavier(width):.3f}')

# check the formula against PyTorch's own implementation
layer = nn.Linear(256, 256, bias=False)
nn.init.xavier_normal_(layer.weight)
expected_std = (2.0 / (256 + 256)) ** 0.5  # nn.init's xavier_normal_ uses fan_in AND fan_out
print(f'\nnn.init.xavier_normal_ weight std: {layer.weight.std().item():.4f}  (formula predicts {expected_std:.4f})')

Xavier održava varijancu u istom približnom rasponu bez obzira na širinu, umjesto katastrofalnog nestajanja. No Xavierov izvod pretpostavlja simetričnu aktivaciju poput `tanh`.

**Kaimingova inicijalizacija** (<a href="../references.html#he-2015-kaiming">He i sur., 2015.</a>) &mdash; poznata i kao *Heova inicijalizacija* &mdash; ista je ideja prilagođena za `ReLU`, koji postavlja približno polovicu ulaza na nulu pa za kompenzaciju zahtijeva dvostruku varijancu: `std = sqrt(2 / fan_in)`. Primjena Xavierova mjerila izvedenog za `tanh` na mrežu s `ReLU` i dalje uzrokuje nestajanje:

In [ ]:
def forward_variance_relu(width, depth=20, kaiming=True):
    x = torch.randn(100, width)
    for _ in range(depth):
        std = (2.0 / width) ** 0.5 if kaiming else (1.0 / width) ** 0.5
        W = torch.randn(width, width) * std
        x = torch.relu(x @ W)
    return x.var().item()

v_xavier_on_relu = forward_variance_relu(256, kaiming=False)
v_kaiming = forward_variance_relu(256, kaiming=True)
print(f'ReLU net, Xavier scale (sqrt(1/fan_in), no ReLU correction): variance after 20 layers = {v_xavier_on_relu:.2e}')
print(f'ReLU net, Kaiming scale (sqrt(2/fan_in)):                    variance after 20 layers = {v_kaiming:.3f}')

layer2 = nn.Linear(256, 256, bias=False)
nn.init.kaiming_normal_(layer2.weight, nonlinearity='relu')
expected_std2 = (2.0 / 256) ** 0.5
print(f'\nnn.init.kaiming_normal_ weight std: {layer2.weight.std().item():.4f}  (formula predicts {expected_std2:.4f})')

Praktično pravilo: koristite Kaimingovu inicijalizaciju za mreže iz obitelji `ReLU` (zadana inicijalizacija za `nn.Conv2d`/`nn.Linear` zapravo je već njezina inačica), a Xavierovu za `tanh`/`sigmoid`. Obje su bolje od odabira fiksne standardne devijacije uz nadu da će raditi. Izvedene su, a ne podešene, iz jednostavnog zahtjeva: održavati varijancu aktivacija približno stalnom iz sloja u sloj, kako bi mreža mogla biti proizvoljno duboka bez tihog gubitka signala prije početka treniranja.

## Rasporedi stope učenja

Jedna fiksna stopa učenja za cijelo treniranje primamljiva je — očita je zadana postavka — ali često je pogrešan odabir: stopa dovoljno velika za brz početni napredak obično je prevelika za precizno smirivanje kada se treniranje približi minimumu. **Rasporedi stope učenja** mijenjaju stopu tijekom vremena. Tri uobičajena rasporeda, provjerena pomoću `torch.optim.lr_scheduler`:

- **Stepenasto smanjivanje**: pomnožite stopu fiksnim faktorom svakih `N` koraka.
- **Kosinusno hlađenje**: glatko smanjujte stopu duž kosinusne krivulje od početne vrijednosti do gotovo nule — naziv je izravno preuzet iz **simuliranog kaljenja**, klasične optimizacijske tehnike koja počinje „vruće” (slobodno istražuje, čak prihvaća i lošije poteze), a zatim se postupno „hladi” prema fiksnom rješenju.
- **Linearno zagrijavanje**: *postupno povećavajte* stopu od nule tijekom prvih nekoliko koraka, prije primjene glavnog rasporeda &mdash; koristi se jer rane promjene, prije stabilizacije statistika gradijenata modela, mogu biti dovoljno nepouzdane da velika stopa od prvog koraka prouzroči štetu koju bi nekoliko koraka postupnog povećavanja izbjeglo.

In [ ]:
base_lr = 0.5

def check_schedule(name, manual_fn, torch_scheduler_factory, steps):
    p = torch.tensor([1.0], requires_grad=True)
    opt = torch.optim.SGD([p], lr=base_lr)
    sched = torch_scheduler_factory(opt)
    torch_lrs = []
    for step in range(steps):
        torch_lrs.append(opt.param_groups[0]['lr'])
        opt.step()
        sched.step()
    manual_lrs = [manual_fn(step) for step in range(steps)]
    max_diff = max(abs(a - b) for a, b in zip(torch_lrs, manual_lrs))
    print(f'{name}: max diff vs torch.optim.lr_scheduler = {max_diff:.2e}')
    return manual_lrs

step_lrs = check_schedule(
    'step decay',
    lambda step: base_lr * (0.5 ** (step // 5)),
    lambda opt: torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5),
    steps=20)

cos_lrs = check_schedule(
    'cosine annealing',
    lambda step: 0.5 * base_lr * (1 + np.cos(np.pi * step / 20)),
    lambda opt: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20),
    steps=20)

warmup_steps = 5
warmup_lrs = check_schedule(
    'linear warmup',
    lambda step: base_lr * min(1.0, (step + 1) / warmup_steps),
    lambda opt: torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=lambda step: min(1.0, (step + 1) / warmup_steps)),
    steps=10)

plt.figure(figsize=(6, 4))
plt.plot(step_lrs, '-o', markersize=3, label='step decay')
plt.plot(cos_lrs, '-o', markersize=3, label='cosine annealing')
plt.plot(warmup_lrs, '-o', markersize=3, label='linear warmup')
plt.xlabel('step'); plt.ylabel('learning rate'); plt.legend(fontsize=8)
plt.title('Three learning rate schedules')
plt.show()

### Zašto smanjivanje doista pomaže: bučni gradijenti

Na točnoj, uskoj zdjeli bez šuma s početka lekcije, dobro odabrana stalna stopa učenja sasvim dobro konvergira — nema razloga za njezino smanjivanje. No, kao što je spomenuto, stvarno treniranje gotovo nikada ne vidi točan gradijent: svaki korak koristi procjenu iz mini-serije, koja sadrži šum. Simuliramo to dodavanjem nasumičnog šuma gradijentu u svakom koraku i uspoređujemo stalnu stopu sa stepenastim smanjivanjem.

In [ ]:
def run_noisy(schedule_fn, steps=400, seed=0, noise_std=0.6):
    rng = np.random.default_rng(seed)
    p = np.array([-4.0, 1.0])
    losses = []
    for t in range(steps):
        noisy_g = grad_f(p) + rng.normal(0, noise_std, size=2)  # stand-in for mini-batch noise
        p = p - schedule_fn(t) * noisy_g
        losses.append(f(p))
    return losses

noisy_lr = 0.18
const_losses = run_noisy(lambda t: noisy_lr)
decay_losses = run_noisy(lambda t: noisy_lr * (0.3 ** (t // 80)))

print(f'constant lr:  mean loss over final 50 steps = {np.mean(const_losses[-50:]):.4f}')
print(f'step decay:   mean loss over final 50 steps = {np.mean(decay_losses[-50:]):.4f}')

plt.figure(figsize=(6, 4))
plt.semilogy(const_losses, label='constant lr', alpha=0.8)
plt.semilogy(decay_losses, label='step decay', alpha=0.8)
plt.xlabel('step'); plt.ylabel('loss (log scale)'); plt.legend(fontsize=8)
plt.title('Noisy gradients: constant lr vs. step decay')
plt.show()

Uz bučne gradijente stalna stopa učenja dovoljno velika za brz početni napredak nikada se stvarno ne smiruje — zauvijek nastavlja poskakivati oko minimuma za iznos proporcionalan samoj stopi učenja. Stepenasto smanjivanje daje isti brz početni napredak, a zatim s nastavkom treniranja smanjuje stopu, sužava poskakivanje i završava mnogo bliže stvarnom minimumu. To je stvarno opravdanje rasporeda stope učenja: fiksna stopa nije „pogrešna” na nekom idealiziranom problemu bez šuma, nego ne može istodobno biti velika (radi brzine) i mala (radi preciznosti) kada je svaki gradijent bučna procjena, što je uobičajeno pri stvarnom treniranju s mini-serijama.

### Zadaci

1. Povećajte `momentum` s 0.9 na 0.99 na problemu uske zdjele. Konvergira li brže ili sada počinje preskakati minimum i oscilirati na *plitkoj* osi?
2. Pokušajte težine MLP-a inicijalizirati identično, ali *vrijednostima različitima od nule* (npr. `np.full((2, H), 0.3)` za oba sloja), umjesto sve na nulu. Ostaje li simetrija i dalje nenarušena? Zašto biste očekivali isti način neuspjeha od svake inicijalizacije u kojoj sve skrivene jedinice počinju identično, a ne samo od inicijalizacije nulama?
3. Adamov `adam_lr` (0.3) mnogo je veći od `lr` običnog SGD-a (0.15) u ovom notebooku, a Adam ipak ostaje stabilan, dok bi obični SGD s `lr=0.3` snažno divergirao na strmoj osi. Isprobajte i potvrdite. Što to sugerira o *razlogu* zbog kojeg se Adam često opisuje kao manje osjetljiv na odabir stope učenja?
4. `forward_variance_relu` uspoređuje Xavierovo i Kaimingovo mjerilo pri `width=256`. Ponovite pri `width=16` i `width=1024`. Raste li ili pada *razlika* između njih s porastom širine i odgovara li to argumentu skaliranja prema broju ulaza korištenom za izvod obiju formula?
5. U demonstraciji stope učenja s bučnim gradijentima isprobajte `noise_std=0.0` (bez ikakvog šuma). Pomaže li stepenasto smanjivanje, odmaže li ili ne donosi razliku u odnosu na stalnu stopu — i odgovara li to tvrdnji da su rasporedi važni *zbog* šuma gradijenta, a ne usprkos samoj površini gubitka?